In [1]:
# import os
# import wandb
# from kaggle_secrets import UserSecretsClient

# # fetch secret
# user_secrets = UserSecretsClient()
# wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# # set environment variable
# os.environ["WANDB_API_KEY"] = wandb_key

# # login
# wandb.login(relogin=True)

In [2]:
!pip install --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 64.4 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.2
    Uninstalling wandb-0.22.2:
      Successfully uninstalled wandb-0.22.2


In [3]:
import os
import wandb

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    
    wandb.login(key=wandb_key)
    print("✓ W&B authenticated via Kaggle secrets")
    
except Exception as e:
    print(f"⚠ W&B login failed: {e}")
    print("Proceeding without W&B logging...")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


✓ W&B authenticated via Kaggle secrets


In [4]:
from pathlib import Path

MESSY_DATA_DIR = Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup")
GENRES_PT = Path("/kaggle/input/datasets/vishalladoiya/pre-compute-spec/genres_pt")
ESC_50_MASTER = MESSY_DATA_DIR / "ESC-50-master"
GENRES_STEMS = MESSY_DATA_DIR / "genres_stems"
MASHUPS = MESSY_DATA_DIR / "mashups"

# Model Architecture  

In [5]:
import torch
import torch.nn as nn
from transformers import ASTForAudioClassification, ASTConfig


class ASTGenreClassifier(nn.Module):
    """
    ASTGenreClassifier

    A fine-tuned Audio Spectrogram Transformer (AST) model for music genre classification.

    This model:
    - Loads a pretrained AST backbone
    - Replaces the default classification head
    - Adapts the model to a custom number of genre classes

    Backbone:
        MIT/ast-finetuned-audioset-10-10-0.4593

    Input:
        Log-Mel Spectrogram tensor of shape:
            [batch_size, 1024, 128]

    Output:
        Logits tensor of shape:
            [batch_size, num_classes]
    """

    def __init__(self, num_classes: int, time_frames: int = 1024, mel_bins: int = 128):
        """
        Initializes the AST-based genre classification model.

        Args:
            num_classes (int): Number of output genre classes.
            time_frames (int): Number of spectrogram time frames (default = 1024).
            mel_bins (int): Number of Mel frequency bins (default = 128).
        """
        super().__init__()

        """
        ASTConfig is used to define the architecture parameters of the model.
        Loading from pretrained ensures we use the same transformer structure
        as the original AudioSet fine-tuned model.
        """
        config = ASTConfig.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593"
        )

        # Update number of output labels
        config.num_labels = num_classes

        """
        ignore_mismatched_sizes=True allows replacing the classifier head
        even if the number of classes differs from the original AudioSet model.
        """
        self.ast = ASTForAudioClassification.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593",
            config=config,
            ignore_mismatched_sizes=True
        )

        self.time_frames = time_frames
        self.mel_bins = mel_bins

        """
        Original AST classifier:
            Linear(hidden_size → original_labels)

        New Custom Head:
            LayerNorm
            Linear(768 → 512)
            GELU activation
            Dropout(0.3)
            Linear(512 → num_classes)
        """
        hidden_size = self.ast.config.hidden_size  # Typically 768

        self.ast.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    @property
    def classifier(self):
        """
        Returns the classifier head of the AST model.
        """
        return self.ast.classifier

    def forward(self, input_values):
        """
        Forward pass of the model.

        Args:
            input_values (torch.Tensor):
                Log-Mel spectrogram tensor.
                Shape:
                    [batch_size, 1024, 128]
                or
                    [batch_size, 1, 1024, 128]

        Returns:
            torch.Tensor:
                Logits of shape [batch_size, num_classes]
        """

        # Remove channel dimension if present
        if input_values.dim() == 4:
            input_values = input_values.squeeze(1)

        outputs = self.ast(input_values=input_values)

        return outputs.logits
    


2026-03-03 03:35:42.453717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772508942.661133      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772508942.719447      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772508943.245246      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772508943.245295      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772508943.245298      23 computation_placer.cc:177] computation placer alr

In [6]:
import torch
import random
from torch.utils.data import Dataset
from pathlib import Path


class Fast_AST_Dataset(Dataset):

    """
    Dataset class for ast model which loads pre computed spectrum from genre_pt folder

    arguments:
        data_dir: path for genre_pt folder 
        train: bool value if true data will be agumented 
    """

    def __init__(self, data_dir: Path, train: bool = True):
        self.data_dir = data_dir
        self.train = train

        self.samples = []       # list of all chunks from data 
        self.labels = []
        self.class_map = {}     # dict for storing index and name of genre
        self.index_map = []   # list of (global_song_idx, chunk_num)

        self._load_files()

        mode = "train" if train else "val"
        print(f"[{mode}] Loaded {len(self.samples)} pre-computed spectrogram chunks.")

    def _load_files(self):
        """
        load all files in samples list
        """

        global_song_counter = 0

        # iterate over upper dir like blue, classical etc
        for label_id, genre_dir in enumerate(sorted(self.data_dir.iterdir())):
            if not genre_dir.is_dir():
                continue

            genre_name = genre_dir.name
            self.class_map[genre_name] = label_id

            # Sort .pt files numerically by chunk number
            chunk_paths = sorted(
                list(genre_dir.glob("*.pt")),
                key=lambda p: int(p.stem.split('_')[1])
            )

            if len(chunk_paths) == 0:
                continue

            chunks_per_song = 3  # matches pre_compute_spec.py (10.24s * 3 ~ 30s song)

            for chunk_path in chunk_paths:
                chunk_num = int(chunk_path.stem.split('_')[1])
                # local song index within this genre
                local_song_idx = chunk_num // chunks_per_song
                # global unique song index across all genres
                global_song_idx = global_song_counter + local_song_idx

                self.samples.append(chunk_path)
                self.labels.append(label_id)
                self.index_map.append((global_song_idx, chunk_num))

            max_chunk = int(chunk_paths[-1].stem.split('_')[1])
            n_songs_this_genre = (max_chunk // chunks_per_song) + 1
            global_song_counter += n_songs_this_genre

    @staticmethod
    def spec_augment(spec: torch.Tensor) -> torch.Tensor:
        """
        spec: [time_frames, mel_bins] = [1024, 128]
        Applies random time and frequency masking in-place on a copy.
        """
        spec = spec.clone()
        T, F = spec.shape  # 1024, 128

        # Frequency masking (up to 20 mel bins masked, 2 masks) 
        for _ in range(2):
            f = random.randint(0, 20)
            if f == 0:
                continue
            f0 = random.randint(0, F - f)
            spec[:, f0:f0 + f] = 0.0

        # Time masking (up to 80 frames masked, 2 masks)
        for _ in range(2):
            t = random.randint(0, 80)
            if t == 0:
                continue
            t0 = random.randint(0, T - t)
            spec[t0:t0 + t, :] = 0.0

        return spec


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        spectrogram, label = torch.load(self.samples[idx], weights_only=True)

        if self.train:
            spectrogram = self.spec_augment(spectrogram)

        return spectrogram, label


def fast_collate_fn(batch):
    """
    Collate pre-computed spectrograms into a batched tensor.
    Input:  list of (spectrogram [1024, 128], label)
    Output: (batch [B, 1024, 128], labels [B])
    """
    spectrograms = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    batched_spectrograms = torch.stack(spectrograms, dim=0)

    clean_labels = [l.item() if isinstance(l, torch.Tensor) else l for l in labels]
    batched_labels = torch.tensor(clean_labels, dtype=torch.long)

    return batched_spectrograms, batched_labels

In [7]:
import os 
import json 
import copy

import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset, Dataset

import numpy as np
import time
import wandb
from scipy.signal import butter, filtfilt

from sklearn.model_selection import StratifiedKFold
from transformers import get_cosine_schedule_with_warmup
from collections import Counter


def highpass_filter(
    audio: torch.Tensor, 
    cutoff: int = 40, 
    sample_rate: int = 16000
):

    """
    Applies a Butterworth high-pass filter to remove very low-frequency noise.

    Args:
        audio (torch.Tensor): Input waveform tensor of shape [1, samples].
        cutoff (int): Cutoff frequency in Hz. Default is 40 Hz.
        sample_rate (int): Sampling rate of the waveform. Default is 16000 Hz.

    Returns:
        torch.Tensor: Filtered waveform with same shape and dtype as input.
    """

    b, a = butter(5, cutoff / (sample_rate / 2), btype='high')

    audio_np = audio.squeeze(0).cpu().numpy()

    filtered = filtfilt(b, a, audio_np)
    filtered = np.ascontiguousarray(filtered)
    filtered = torch.from_numpy(filtered).unsqueeze(0)

    return filtered.type_as(audio)



def mixup_data(
    x: torch.Tensor, 
    y: torch.Tensor, 
    alpha: float = 0.4
):
    """Returns mixed inputs, pairs of targets, and lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(
    criterion, 
    pred, 
    y_a, 
    y_b,
    lam
):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def _print_trainable(model, current_epoch, total_epochs):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"        Trainable: {trainable:,} / {total:,}")
    print(f"{'=' * 70}\n")


def get_llrd_optimizer(
    model: nn.Module, 
    base_lr: float = 1e-5, 
    decay_factor: float = 0.65,
    weight_decay: float = 0.05
):
    """
    Layer-wise Learning Rate Decay (LLRD) with strict parameter tracking.
    """
    param_groups = []
    assigned_params = set()

    # 1. Classifier head — highest LR
    classifier_params = []
    for n, p in model.named_parameters():
        if "classifier" in n and p.requires_grad:
            classifier_params.append(p)
            assigned_params.add(p)
    
    if classifier_params:
        param_groups.append({
            'params': classifier_params,
            'lr': base_lr * 10,
            'name': 'classifier'
        })

    # 2. Transformer layers — decaying LR from top (11) to bottom (0)
    num_layers = 12
    for layer_idx in range(num_layers - 1, -1, -1):
        lr = base_lr * (decay_factor ** (num_layers - 1 - layer_idx))
        layer_params = []
        for n, p in model.named_parameters():
            if f"encoder.layer.{layer_idx}." in n and p.requires_grad and p not in assigned_params:
                layer_params.append(p)
                assigned_params.add(p)
        
        if layer_params:
            param_groups.append({
                'params': layer_params,
                'lr': lr,
                'name': f'encoder_layer_{layer_idx}'
            })

    # 3. Embeddings — lowest LR
    embed_params = []
    for n, p in model.named_parameters():
        if 'embeddings' in n and p.requires_grad and p not in assigned_params:
            embed_params.append(p)
            assigned_params.add(p)
    
    if embed_params:
        param_groups.append({
            'params': embed_params,
            'lr': base_lr * 0.1,
            'name': 'embeddings'
        })

    # 4. Catch-all for unassigned parameters (like AST's final layernorm)
    other_params = []
    for n, p in model.named_parameters():
        if p.requires_grad and p not in assigned_params:
            other_params.append(p)
            assigned_params.add(p)
    
    if other_params:
        param_groups.append({
            'params': other_params,
            'lr': base_lr,
            'name': 'other_components'
        })

    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


def train_ast(
    model: nn.Module,
    dataloader: DataLoader, 
    loss_function: nn.Module, 
    optimizer: torch.optim.Optimizer, 
    device: str,
    use_mixup:bool=True, 
    mixup_alpha:float=0.4, 
    scaler: nn.Module=None, 
    grad_accum_steps: int=1,
    scheduler: torch.optim.lr_scheduler._LRScheduler = None
):
    """
    Training loop with three memory-saving techniques for Stage 3:
      - Mixed-precision (AMP): halves activation memory automatically
      - Gradient accumulation: lets you simulate a larger batch with less VRAM
      - Gradient checkpointing: trades compute for memory in the encoder
        (enabled on the model before calling this function)
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()

    for step, (input_values, labels) in enumerate(dataloader):
        input_values = input_values.to(device)
        labels = labels.to(device)

        # forward (with AMP if scaler provided)
        with autocast("cuda", enabled=(scaler is not None)):
            if use_mixup:
                input_values, labels_a, labels_b, lam = mixup_data(input_values, labels, alpha=mixup_alpha)
                logits = model(input_values)
                loss = mixup_criterion(loss_function, logits, labels_a, labels_b, lam)
            else:
                logits = model(input_values)
                loss = loss_function(logits, labels)

            # Scale loss for gradient accumulation
            loss: torch.Tensor = loss / grad_accum_steps

        if torch.isnan(loss):
            print("nan loss detected, skipping batch")
            optimizer.zero_grad()
            continue

        # backward
        if scaler is not None:
            """
            dynamically estimates the scale factor each iteration
            Helps perform the steps of gradient scaling
            conveniently.

            scaler.scale()
            multiplies a given loss by ``scaler``'s current scale factor.
            """
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # optimizer step every grad_accum_steps batches
        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(dataloader):

            if scaler is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            optimizer.zero_grad()

            if scheduler is not None:
                scheduler.step()

        running_loss += loss.item() * grad_accum_steps   # un-scale for logging

        _, predicted = torch.max(logits, 1)

        ref_labels = labels if not use_mixup else labels_a

        correct += (predicted == ref_labels).sum().item()
        total += ref_labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total
    return epoch_loss, accuracy


def test_ast(
    model: nn.Module, 
    dataloader: DataLoader, 
    loss_function: torch.optim.Optimizer, 
    device: str
):
    """Chunk-level evaluation (fast, used every epoch)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for input_values, labels in dataloader:
            input_values = input_values.to(device)
            labels = labels.to(device)

            logits = model(input_values)
            loss = loss_function(logits, labels)

            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0
    accuracy = 100 * correct / total if total > 0 else 0
    return loss, accuracy


def test_ast_song_level(
    model: nn.Module, 
    dataset_subset: Subset, 
    original_dataset: Dataset, 
    device: str
):
    """
    Song-level evaluation using majority voting across all chunks of a song.

    Args:
        dataset_subset : torch Subset wrapping Fast_AST_Dataset (val split)
        original_dataset: the underlying Fast_AST_Dataset (has index_map + labels)
    Returns:
        song_accuracy (float)
    """
    model.eval()

    # Collect per-chunk predictions grouped by song_idx
    song_preds = {}   # song_idx -> list of predicted class ints
    song_labels = {}  # song_idx -> true label

    with torch.no_grad():
        for i, real_idx in enumerate(dataset_subset.indices):
            spec, label = original_dataset[real_idx]
            spec = spec.unsqueeze(0).to(device)

            logits = model(spec)
            pred = torch.argmax(logits, dim=1).item()
            label_val = label.item() if isinstance(label, torch.Tensor) else label

            song_idx = original_dataset.index_map[real_idx][0]

            if song_idx not in song_preds:
                song_preds[song_idx] = []
                song_labels[song_idx] = label_val

            song_preds[song_idx].append(pred)

    # Majority vote
    correct = 0
    for song_idx, preds in song_preds.items():
        voted = max(set(preds), key=preds.count)
        if voted == song_labels[song_idx]:
            correct += 1

    return 100 * correct / len(song_preds) if song_preds else 0


def train_ast_model(
    model: nn.Module, 
    device: str, 
    train_loader: DataLoader, 
    val_loader: DataLoader,
    save_path: str, 
    val_dataset_subset: Subset=None, 
    original_dataset: Subset=None,
    epochs: int=30
):
    
    """
    Trains one model with the 3-stage progressive unfreezing strategy.

      - weight_decay 1e-4 → 0.05  (proper transformer regularisation)
      - LR warmup added to every stage  (prevents instability)
      - Stage 2 unfreezes layers 6-11 + layernorm  (was 8-11 only)
      - Stage 3 uses LLRD instead of flat LR
      - Scheduler stepped per-batch (required for warmup)
      - Mixup active from epoch 0 (not just stage 2+)
      - Stage boundaries: 20 / 65 split  (more time for full fine-tune)
    """

    use_cuda = (device == "cuda")
    if use_cuda:
        torch.cuda.empty_cache()

    # AMP scaler — only meaningful on CUDA
    scaler = GradScaler() if use_cuda else None

    model = model.to(device)

    loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)

    start_time = time.time()
    best_val_acc = 0.0
    best_val_loss = float('inf')

    # Stage boundaries — 30 / 50 / 20 split
    stage2_start = int(epochs * 0.30)
    stage3_start = int(epochs * 0.80)

    optimizer = None
    scheduler = None
    grad_accum_steps = 1   
    mixup_alpha = 0.4       

    for epoch in range(epochs):
        epoch_start = time.time()

        if epoch == 0:
            print(f"STAGE 1: Classifier head only  (lr=1e-3, epochs 1-{stage2_start})")

            for param in model.parameters():
                param.requires_grad = False
            for param in model.classifier.parameters():
                param.requires_grad = True

            grad_accum_steps = 1
            mixup_alpha = 0.4  

            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=1e-3, weight_decay=1e-4
            )

            total_steps = (stage2_start * len(train_loader)) // grad_accum_steps
            warmup_steps = (2 * len(train_loader)) // grad_accum_steps 

            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )

            scheduler.step()

            _print_trainable(model, epoch + 1, epochs)
            wandb.log({"stage": 1}, step=epoch)

        elif epoch == stage2_start:
            print(f"\n{'=' * 70}")
            print(f"STAGE 2: Layers 8-11 unfrozen  (lr=5e-5, epochs {stage2_start+1}-{stage3_start})")

            for name, param in model.named_parameters():
                for layer in ["encoder.layer.6", "encoder.layer.7", "encoder.layer.8", "encoder.layer.9", "encoder.layer.10", "encoder.layer.11","layernorm"]:
                    if layer in name:
                        param.requires_grad = True

            grad_accum_steps = 1
            mixup_alpha = 0.2  

            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=5e-5, weight_decay=1e-4
            )

            n_steps = ((stage3_start - stage2_start) * len(train_loader)) // grad_accum_steps
            warmup_steps = (1 * len(train_loader)) // grad_accum_steps

            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=n_steps
            )

            scheduler.step()

            _print_trainable(model, epoch + 1, epochs)
            wandb.log({"stage": 2}, step=epoch)

        elif epoch == stage3_start:
            print(f"\n{'=' * 70}")
            print(f"STAGE 3: Full fine-tune  (lr=1e-5, epochs {stage3_start+1}-{epochs})")
            print(f"         [AMP + gradient checkpointing + grad accum x4]")

            # Unfreeze everything except embeddings
            for name, param in model.named_parameters():
                if "embeddings" not in name:
                    param.requires_grad = True

            grad_accum_steps = 4
            mixup_alpha = 0.2

            if hasattr(model, 'ast'):
                model.ast.gradient_checkpointing_enable()
            elif hasattr(model, 'model'):
                model.model.gradient_checkpointing_enable()
            elif hasattr(model, 'encoder'):
                model.encoder.gradient_checkpointing_enable()
            elif hasattr(model, 'base_model'):
                model.base_model.gradient_checkpointing_enable()
            else:
                print("⚠️ Warning: Could not find the internal AST model to enable gradient checkpointing. Continuing without it.")

            grad_accum_steps = 4

            optimizer = get_llrd_optimizer(model, base_lr=1e-5, decay_factor=0.65)
            
            n_steps = ((epochs - stage3_start) * len(train_loader)) // grad_accum_steps
            warmup_steps = (1 * len(train_loader)) // grad_accum_steps

            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=n_steps
            )

            scheduler.step()

            if use_cuda:
                torch.cuda.empty_cache()

            _print_trainable(model, epoch + 1, epochs)
            wandb.log({"stage": 3}, step=epoch)

        use_mixup = (epoch >= stage2_start)

        train_loss, train_acc = train_ast(
            model=model,
            dataloader=train_loader, 
            loss_function=loss_function, 
            optimizer=optimizer, 
            device=device,
            use_mixup=use_mixup,
            mixup_alpha=mixup_alpha,
            scaler=scaler, 
            grad_accum_steps=grad_accum_steps
        )
        val_loss, val_acc_chunk = test_ast(
            model=model, 
            dataloader=val_loader, 
            loss_function=loss_function, 
            device=device
        )

        # Song-level majority vote (only if caller provides the subset + dataset)
        val_acc_song = None
        # if val_dataset_subset is not None and original_dataset is not None:
        val_acc_song = test_ast_song_level(
            model=model,
            dataset_subset=val_dataset_subset, original_dataset=original_dataset, 
            device=device
        )


        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]

        log_dict = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy_chunk": val_acc_chunk,
            "learning_rate": current_lr,
            "epoch_time": epoch_time,
        }
        if val_acc_song is not None:
            log_dict["val_accuracy_song"] = val_acc_song

        wandb.log(log_dict)

        song_str = f" | Song Acc: {val_acc_song:6.2f}%" if val_acc_song is not None else ""
        print(
            f"Epoch [{epoch+1:3d}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:6.2f}% | "
            f"Val Loss: {val_loss:.4f} | Chunk Acc: {val_acc_chunk:6.2f}%"
            f"{song_str} | LR: {current_lr:.2e} | Time: {epoch_time:.1f}s"
        )

        # Save on best chunk-level val acc (or song-level if available)
        primary_acc = val_acc_song if val_acc_song is not None else val_acc_chunk

        if primary_acc > best_val_acc:
            best_val_acc = primary_acc
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            metric_name = "song" if val_acc_song is not None else "chunk"
            print(f"    ✓ Saved [{metric_name}: {primary_acc:.2f}%]  →  {save_path}")

    total_time = time.time() - start_time

    print(f"\n{'=' * 70}")
    print(f"AST TRAINING COMPLETE  (v2)")
    print(f"{'=' * 70}")
    print(f"Total Training Time    : {total_time / 60:.1f} minutes")
    print(f"Best Validation Acc    : {best_val_acc:.2f}%")
    print(f"Best Validation Loss   : {best_val_loss:.4f}")
    print(f"Model saved as         : best_ast_genre.pth")
    print(f"{'=' * 70}\n")

    wandb.finish()
    return best_val_acc

def train_single_corss_validation(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    val_dataset_subset: Subset = None,
    original_dataset: Subset = None,
    epochs: int = 30,
    save_dir: str = "."
):
    print("\n" + "=" * 70)
    print("AST GENRE CLASSIFICATION — SINGLE RUN")
    print("=" * 70 + "\n")

    use_cuda = device == "cuda"
    if use_cuda:
        torch.cuda.empty_cache()

    wandb.init(
        project="ast-genre-classification-signel-run",
        name="3-stage-v4-mixup-specaug-amp",
        config={
            "epochs": epochs,
            "optimizer": "AdamW",
            "mixup_alpha": 0.4,
            "label_smoothing": 0.1,
            "spec_augment": True,
            "amp": use_cuda,
            "grad_checkpointing_stage3": True,
            "grad_accum_stage3": 4,
            "scheduler": "CosineAnnealingLR",
            "dataset": "Genre",
            "architecture": model.__class__.__name__,
            "base_model": "ast-finetuned-audioset-10-10-0.4593",
        }
    )
    wandb.watch(model, log="all", log_freq=50)

    model = model.to(device)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "best_ast_genre.pth")

    best_acc = train_ast_model(
        model=model,
        device=device,
        train_loader=train_loader,
        val_loader=val_loader,
        original_dataset=original_dataset,
        val_dataset_subset=val_dataset_subset,
        save_path=save_path,
        epochs=epochs
    )

    wandb.finish()
    return best_acc


def train_ast_cross_validation(
    model_fn: nn.Module,
    device: str,
    full_dataset: Subset,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_folds: int = 5, 
    epochs: int = 30,
    save_dir: str = "cv_models",
    wandb_project: str = "ast-genre-classification-cross-validation",
):
    """
    Stratified K-Fold cross-validation.

    Each fold trains a completely independent model, saved to:
        {save_dir}/fold_{k}/best_ast_genre_fold{k}.pth

    A metadata JSON is also saved:
        {save_dir}/cv_results.json

    These saved models can later be loaded together for ensemble inference
    """

    os.makedirs(save_dir, exist_ok=True)

    all_labels = np.array(full_dataset.labels)
    all_indices = np.arange(len(full_dataset))

    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=42
    )

    fold_results = []

    print("\n" + "=" * 70)
    print(f"AST GENRE CLASSIFICATION — {n_folds}-FOLD CROSS-VALIDATION")
    print("=" * 70)
    print(f"  Dataset size : {len(full_dataset)} chunks")
    print(f"  Folds        : {n_folds}")
    print(f"  Epochs/fold  : {epochs}")
    print(f"  Save dir     : {save_dir}")
    print("=" * 70 + "\n")

    cv_start = time.time()

    for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(all_indices, all_labels)
    ):
        fold_num = fold_idx + 1 

        print(f"\n{'═' * 70}")
        print(f"  FOLD {fold_num}/{n_folds}  |  train={len(train_idx)}  val={len(val_idx)}")
        print(f"{'═' * 70}\n")

        fold_dir = os.path.join(save_dir, f"fold_{fold_num}")
        os.makedirs(fold_dir, exist_ok=True)

        save_path = os.path.join(fold_dir, f"best_ast_genre_fold{fold_num}.pth")

        train_subset = Subset(full_dataset, train_idx)
        val_subset   = Subset(full_dataset, val_idx)

        model = model_fn.to(device)

        run = wandb.init(
            project=wandb_project,
            name=f"fold-{fold_num}-of-{n_folds}",
            group="cross-validation",
            config={
                "fold": fold_num, "n_folds": n_folds,
                "epochs": epochs, "optimizer": "AdamW",
                "label_smoothing": 0.1,
                "scheduler": "cosine-with-warmup",
                "architecture": model.__class__.__name__,
            },
            reinit=True,
        )

        fold_start = time.time()

        best_acc = train_single_corss_validation(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            val_dataset_subset=val_subset,
            original_dataset=full_dataset,
            epochs=epochs,
            save_dir=save_path,
            device=device
        )

        fold_time = time.time() - fold_start

        result = {
            "fold": fold_num,
            "best_val_acc": round(best_acc, 4),
            "model_path": save_path,
            "train_size": len(train_idx),
            "val_size": len(val_idx),
            "epochs": epochs,
            "time_minutes": round(fold_time / 60, 2),
        }
        fold_results.append(result)

        print(f"\n  Fold {fold_num} done  |  Best acc: {best_acc:.2f}%  |  {fold_time/60:.1f} min")
        wandb.finish()

    accs = [r["best_val_acc"] for r in fold_results]
    cv_summary = {
        "mean_acc": round(float(np.mean(accs)), 4),
        "std_acc": round(float(np.std(accs)), 4),
        "max_acc": round(float(np.max(accs)), 4),
        "min_acc": round(float(np.min(accs)), 4),
        "fold_results": fold_results,
        "total_time_minutes": round((time.time() - cv_start) / 60, 2),
    }

    results_path = os.path.join(save_dir, "cv_results.json")
    with open(results_path, "w") as f:
        json.dump(cv_summary, f, indent=2)

    print(f"\n{'=' * 70}")
    print(f"CROSS-VALIDATION COMPLETE")
    print(f"{'=' * 70}")
    print(f"  Mean acc : {cv_summary['mean_acc']:.2f}% ± {cv_summary['std_acc']:.2f}%")
    print(f"  Best fold: {cv_summary['max_acc']:.2f}%")
    print(f"  Results  : {results_path}")
    for r in fold_results:
        print(f"    Fold {r['fold']}: {r['best_val_acc']:.2f}%  →  {r['model_path']}")
    print(f"{'=' * 70}\n")

    return fold_results



In [8]:
# from ast_model import ASTGenreClassifier 
# from utils import train_ast_cross_validation 
# from ast_dataset import Fast_AST_Dataset, fast_collate_fn

import torch
from torch.utils.data import Subset, WeightedRandomSampler, DataLoader, Dataset
from pathlib import Path
from collections import Counter


class Pipeline:
    def __init__(
            self, 
            dataset_path: Path,         
            batch_size: int = 16, 
            num_workers: int = 4,
            num_classes: int = 10,
            epochs: int = 30,
            n_fold: int = 5
        ):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.batch_size= batch_size
        self.num_workers = num_workers
        self.epochs = epochs
        self.n_fold = n_fold

        self.base_model = ASTGenreClassifier(num_classes=num_classes)

        self.train_dataloader, self.val_dataloader, self.train_full_dataset = self.load_data(dataset_path=dataset_path)

    def load_data(self, dataset_path: Path) -> tuple[DataLoader, DataLoader, Dataset]:

        train_full_dataset = Fast_AST_Dataset(data_dir=dataset_path, train=True)
        val_full_dataset = Fast_AST_Dataset(data_dir=dataset_path, train=False) 

        unique_song_ids = list(set([item[0] for item in train_full_dataset.index_map]))
        num_songs = len(unique_song_ids)

        train_song_count = int(0.8 * num_songs)
        train_songs = set(unique_song_ids[:train_song_count])
        val_songs = set(unique_song_ids[train_song_count:])

        print(f'\nSongs split: {len(train_songs)} train, {len(val_songs)} val')

        train_indices = [
            i for i, (song_idx, _) in enumerate(train_full_dataset.index_map) 
            if song_idx in train_songs
        ]

        val_indices = [
            i for i, (song_idx, _) in enumerate(val_full_dataset.index_map) 
            if song_idx in val_songs
        ]

        train_dataset = Subset(train_full_dataset, train_indices)
        val_dataset = Subset(val_full_dataset,   val_indices)

        print(f'Train chunks: {len(train_dataset)}')
        print(f'Val chunks:   {len(val_dataset)}')

        train_labels = [train_full_dataset.labels[i] for i in train_indices]
        class_counts = Counter(train_labels)
        class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

        print('\nClass distribution in training chunks:')
        genre_names = {v: k for k, v in train_full_dataset.class_map.items()}
        for cls, count in sorted(class_counts.items()):
            print(f'  {genre_names[cls]}: {count} chunks  (weight: {class_weights[cls]:.5f})')

        train_dataloader = DataLoader(
            dataset=train_dataset,
            batch_size=self.batch_size,
            sampler=sampler,
            collate_fn=fast_collate_fn,
            num_workers=self.num_workers,
            pin_memory=(self.device == 'cuda')
        )

        val_dataloader = DataLoader(
            dataset=val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=fast_collate_fn,
            num_workers=self.num_workers,
            pin_memory=(self.device == 'cuda')
        )

        return train_dataloader, val_dataloader, train_full_dataset

    def cross_validation(self):
        fold_results = train_ast_cross_validation(
            model_fn=self.base_model,
            device=self.device,
            full_dataset=self.train_full_dataset,
            train_loader=self.train_dataloader,
            val_loader=self.val_dataloader,
            epochs=self.epochs,
        )

        print(fold_results)
        return fold_results
    

In [9]:
modle_pipeline = Pipeline(dataset_path=GENRES_PT)
modle_pipeline.cross_validation()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[train] Loaded 3000 pre-computed spectrogram chunks.
[val] Loaded 3000 pre-computed spectrogram chunks.

Songs split: 800 train, 200 val
Train chunks: 2400
Val chunks:   600

Class distribution in training chunks:
  blues: 300 chunks  (weight: 0.00333)
  classical: 300 chunks  (weight: 0.00333)
  country: 300 chunks  (weight: 0.00333)
  disco: 300 chunks  (weight: 0.00333)
  hiphop: 33 chunks  (weight: 0.03030)
  jazz: 156 chunks  (weight: 0.00641)
  metal: 300 chunks  (weight: 0.00333)
  pop: 117 chunks  (weight: 0.00855)
  reggae: 300 chunks  (weight: 0.00333)
  rock: 294 chunks  (weight: 0.00340)

AST GENRE CLASSIFICATION — 5-FOLD CROSS-VALIDATION
  Dataset size : 3000 chunks
  Folds        : 5
  Epochs/fold  : 30
  Save dir     : cv_models


══════════════════════════════════════════════════════════════════════
  FOLD 1/5  |  train=2400  val=600
══════════════════════════════════════════════════════════════════════



wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run qygj36oo
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260303_033600-qygj36oo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run fold-1-of-5
wandb: ⭐️ View project at https://wandb.ai/23f2003086-t12026-project/ast-genre-classification-cross-validation
wandb: 🚀 View run at https://wandb.ai/23f2003086-t12026-project/ast-genre-classification-cross-validation/runs/qygj36oo
wandb: Finishing previous runs because reinit is set to 'default'.



AST GENRE CLASSIFICATION — SINGLE RUN



... (27 outputs left out)

STAGE 1: Classifier head only  (lr=1e-3, epochs 1-9)
        Trainable: 400,394 / 86,587,658

Epoch [  1/30] | Train Loss: 0.8223 | Train Acc:  88.38% | Val Loss: 0.9209 | Chunk Acc:  84.33% | Song Acc:  85.20% | LR: 3.33e-06 | Time: 241.8s
    ✓ Saved [song: 85.20%]  →  cv_models/fold_5/best_ast_genre_fold5.pth/best_ast_genre.pth
Epoch [  2/30] | Train Loss: 0.8528 | Train Acc:  86.29% | Val Loss: 0.9209 | Chunk Acc:  83.83% | Song Acc:  85.00% | LR: 3.33e-06 | Time: 242.5s
Epoch [  3/30] | Train Loss: 0.8265 | Train Acc:  88.38% | Val Loss: 0.9315 | Chunk Acc:  83.67% | Song Acc:  84.60% | LR: 3.33e-06 | Time: 241.6s
Epoch [  4/30] | Train Loss: 0.8491 | Train Acc:  86.54% | Val Loss: 0.9314 | Chunk Acc:  83.67% | Song Acc:  83.80% | LR: 3.33e-06 | Time: 243.9s
Epoch [  5/30] | Train Loss: 0.8326 | Train Acc:  87.50% | Val Loss: 0.9304 | Chunk Acc:  83.67% | Song Acc:  85.40% | LR: 3.33e-06 | Time: 243.0s
    ✓ Saved [song: 85.40%]  →  cv_models/fold_5/best_ast_genre_fold5.pth/best_a

wandb: updating run metadata
wandb: uploading history steps 29-29, summary, console lines 49-58
wandb: 
wandb: Run history:
wandb:              epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:         epoch_time ▃▄▃▆▅▄▆▃▄▅▁▆▃▆▆▂▄▄▇█▃▄▆▇▆▂▅▅▅▄
wandb:      learning_rate █████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▇▇▇▇▇
wandb:              stage ▁▅█
wandb:     train_accuracy █████████▂▃▂▂▁▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▂
wandb:         train_loss ▁▂▁▂▁▁▁▁▁▇▇▆▆▇█▆▆▅▆▇▇▆█▇▇▇▆▇▇▇
wandb: val_accuracy_chunk ▅▂▁▁▁▁▂▁▂▃▄▅▅▅▅▅▆▆▆▇▇▇██▇█▇▇█▇
wandb:  val_accuracy_song ▆▆▅▃▆▃▆▆██▆▂▆▃█▁▅▄▄▂▃▂▆▃▂▆▅▂▆▄
wandb:           val_loss ▃▃▆▆▆█▅▆▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▂▁
wandb: 
wandb: Run summary:
wandb:              epoch 30
wandb:         epoch_time 242.68991
wandb:      learning_rate 0.0
wandb:              stage 3
wandb:     train_accuracy 49.125
wandb:         train_loss 1.10054
wandb: val_accuracy_chunk 84.83333
wandb:  val_accuracy_song 84.2
wandb:           val_loss 0.91674
wandb: 
wandb: 🚀 View run 3-stage-v4-mixup-specaug-amp at: https://wandb


  Fold 5 done  |  Best acc: 86.20%  |  121.6 min

CROSS-VALIDATION COMPLETE
  Mean acc : 84.75% ± 1.82%
  Best fold: 86.72%
  Results  : cv_models/cv_results.json
    Fold 1: 81.74%  →  cv_models/fold_1/best_ast_genre_fold1.pth
    Fold 2: 83.71%  →  cv_models/fold_2/best_ast_genre_fold2.pth
    Fold 3: 85.37%  →  cv_models/fold_3/best_ast_genre_fold3.pth
    Fold 4: 86.72%  →  cv_models/fold_4/best_ast_genre_fold4.pth
    Fold 5: 86.20%  →  cv_models/fold_5/best_ast_genre_fold5.pth

[{'fold': 1, 'best_val_acc': 81.7427, 'model_path': 'cv_models/fold_1/best_ast_genre_fold1.pth', 'train_size': 2400, 'val_size': 600, 'epochs': 30, 'time_minutes': 137.85}, {'fold': 2, 'best_val_acc': 83.7113, 'model_path': 'cv_models/fold_2/best_ast_genre_fold2.pth', 'train_size': 2400, 'val_size': 600, 'epochs': 30, 'time_minutes': 96.09}, {'fold': 3, 'best_val_acc': 85.3659, 'model_path': 'cv_models/fold_3/best_ast_genre_fold3.pth', 'train_size': 2400, 'val_size': 600, 'epochs': 30, 'time_minutes': 104

[{'fold': 1,
  'best_val_acc': 81.7427,
  'model_path': 'cv_models/fold_1/best_ast_genre_fold1.pth',
  'train_size': 2400,
  'val_size': 600,
  'epochs': 30,
  'time_minutes': 137.85},
 {'fold': 2,
  'best_val_acc': 83.7113,
  'model_path': 'cv_models/fold_2/best_ast_genre_fold2.pth',
  'train_size': 2400,
  'val_size': 600,
  'epochs': 30,
  'time_minutes': 96.09},
 {'fold': 3,
  'best_val_acc': 85.3659,
  'model_path': 'cv_models/fold_3/best_ast_genre_fold3.pth',
  'train_size': 2400,
  'val_size': 600,
  'epochs': 30,
  'time_minutes': 104.58},
 {'fold': 4,
  'best_val_acc': 86.7203,
  'model_path': 'cv_models/fold_4/best_ast_genre_fold4.pth',
  'train_size': 2400,
  'val_size': 600,
  'epochs': 30,
  'time_minutes': 113.23},
 {'fold': 5,
  'best_val_acc': 86.2,
  'model_path': 'cv_models/fold_5/best_ast_genre_fold5.pth',
  'train_size': 2400,
  'val_size': 600,
  'epochs': 30,
  'time_minutes': 121.59}]

# prediction or inference code

Predict the music genre of an audio file.

      - All chunks are processed in batches (faster, avoids repeated overhead)
      - Combines BOTH soft-vote (avg probabilities) and hard-vote (majority)
        and uses soft-vote as the primary result (more accurate)
      - Returns both voting results so callers can inspect agreement
      - Graceful handling of very short files (< 1 chunk)

In [10]:
import torch
from pathlib import Path
import torchaudio
import numpy as np
from transformers import ASTFeatureExtractor
from scipy.signal import butter, filtfilt

feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

genre_map = {
    'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
    'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9
}


def highpass_filter(audio: torch.Tensor, cutoff: int = 40, sample_rate: int = 16000):
    b, a = butter(5, cutoff / (sample_rate / 2), btype='high')

    audio_np = audio.squeeze(0).cpu().numpy()
    filtered = filtfilt(b, a, audio_np)

    filtered = np.ascontiguousarray(filtered)
    filtered = torch.from_numpy(filtered).unsqueeze(0)
    return filtered.type_as(audio)


def _load_and_preprocess(audio_path, sample_rate: int = 16000) -> torch.Tensor:
    """Load audio, convert to mono, resample, and apply highpass filter."""
    waveform, sr = torchaudio.load(audio_path)
    waveform = waveform.mean(dim=0, keepdim=True)

    if sr != sample_rate:
        resampler = torchaudio.transforms.Resample(sr, sample_rate)
        waveform = resampler(waveform)

    waveform = highpass_filter(waveform, sample_rate=sample_rate)
    return waveform


def _chunk_waveform(waveform: torch.Tensor, chunk_samples: int) -> list[torch.Tensor]:
    """
    Split waveform into non-overlapping chunks of exactly chunk_samples.
    The last chunk is zero-padded if shorter.
    Always returns at least 1 chunk.
    """
    total_samples = waveform.shape[1]
    n_chunks = max(1, total_samples // chunk_samples)
    chunks = []

    for c in range(n_chunks):
        start = c * chunk_samples
        end = start + chunk_samples

        if end > total_samples:
            chunk = waveform[:, start:]
            padding = chunk_samples - chunk.shape[1]
            if padding > 0:
                chunk = torch.nn.functional.pad(chunk, (0, padding))
        else:
            chunk = waveform[:, start:end]

        chunks.append(chunk)

    return chunks


def _normalize_chunk(chunk: torch.Tensor) -> torch.Tensor:
    """RMS normalization"""
    rms = torch.sqrt(torch.mean(chunk ** 2) + 1e-9)
    chunk = chunk / (rms * 3.0)
    return chunk.clamp(-1.0, 1.0)


def _chunks_to_spectrograms(chunks: list[torch.Tensor], sample_rate: int) -> torch.Tensor:
    """
    Convert a list of 1-D waveform chunks to a batched spectrogram tensor.
    Returns shape: [N, 1024, 128]
    Processing the whole song in one feature_extractor call is faster and
    ensures identical padding behaviour across all chunks.
    """
    numpy_chunks = [c.squeeze(0).cpu().numpy() for c in chunks]

    inputs = feature_extractor(
        numpy_chunks,
        sampling_rate=sample_rate,
        padding="max_length",
        return_tensors="pt"
    )
    return inputs.input_values   # [N, 1024, 128]


@torch.no_grad()
def predict_ast(model, audio_path, device, genre_map,
                batch_size: int = 8) -> tuple[str, float, dict]:
    """
    Predict the music genre of an audio file.
      - All chunks are processed in batches (faster, avoids repeated overhead)
      - Combines BOTH soft-vote (avg probabilities) and hard-vote (majority)
        and uses soft-vote as the primary result (more accurate)
      - Returns both voting results so callers can inspect agreement
      - Graceful handling of very short files (< 1 chunk)

    Args:
        model      : trained ASTGenreClassifier
        audio_path : path to the audio file
        device     : "cuda" or "cpu"
        genre_map  : dict mapping genre_name -> class_index
        batch_size : number of chunks to forward through model at once

    Returns:
        predicted_genre : str
        confidence      : float  (soft-vote probability of winning class)
        all_probs       : dict   genre_name -> probability (soft-vote)
    """
    model.eval()

    SAMPLE_RATE   = 16000
    CHUNK_SAMPLES = int(SAMPLE_RATE * 10.24)   # 163_840 samples = 1024 spectrogram frames

    index_to_genre = {v: k for k, v in genre_map.items()}

    waveform = _load_and_preprocess(audio_path, SAMPLE_RATE)

    chunks = _chunk_waveform(waveform, CHUNK_SAMPLES)
    chunks = [_normalize_chunk(c) for c in chunks]

    spectrograms = _chunks_to_spectrograms(chunks, SAMPLE_RATE)  # [N, 1024, 128]

    all_chunk_probs = []

    for i in range(0, len(spectrograms), batch_size):
        batch = spectrograms[i:i + batch_size].to(device)
        logits = model(batch)
        probs  = torch.softmax(logits, dim=1)          # [B, num_classes]
        all_chunk_probs.append(probs.cpu())

    all_chunk_probs = torch.cat(all_chunk_probs, dim=0)  # [N, num_classes]

    # Soft vote — average probabilities, then argmax
    avg_probs       = all_chunk_probs.mean(dim=0)          # [num_classes]
    soft_pred_idx   = avg_probs.argmax().item()
    soft_confidence = avg_probs[soft_pred_idx].item()

    # Hard vote — majority voting across chunk argmaxes
    chunk_preds    = all_chunk_probs.argmax(dim=1).tolist()
    hard_pred_idx  = max(set(chunk_preds), key=chunk_preds.count)
    hard_votes     = chunk_preds.count(hard_pred_idx)

    predicted_genre = index_to_genre[soft_pred_idx]
    hard_genre      = index_to_genre[hard_pred_idx]

    all_probs = {
        index_to_genre[i]: prob.item()
        for i, prob in enumerate(avg_probs)
    }

    # Sort by probability descending for readability
    all_probs_sorted = dict(
        sorted(all_probs.items(), key=lambda x: x[1], reverse=True)
    )

    # Log agreement between voting methods (useful for debugging)
    agreement = "✅ agree" if soft_pred_idx == hard_pred_idx else \
                f"⚠️  disagree (hard vote → {hard_genre} with {hard_votes}/{len(chunks)} chunks)"

    print(f"[predict] {Path(audio_path).name}")
    print(f"  Chunks processed : {len(chunks)}")
    print(f"  Soft vote        : {predicted_genre}  ({soft_confidence:.1%})")
    print(f"  Hard vote        : {hard_genre}  ({hard_votes}/{len(chunks)} chunks)  {agreement}")
    print(f"  Top-3            : " + "  |  ".join(
        f"{g}: {p:.1%}" for g, p in list(all_probs_sorted.items())[:3]
    ))

    return predicted_genre, soft_confidence, all_probs_sorted

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [11]:
# import torch 
# device = "cuda" if torch.cuda.is_available() else "cpu"

# num_classes = 10
# model = ASTGenreClassifier(num_classes)
# model.load_state_dict(torch.load("/kaggle/input/models/vishalladoiya/best-ast-model/pytorch/default/1/best_ast_genre_s3_v3.pth", map_location=device))
# model = model.to(device)